In [ ]:
import pandas as pd
import os
import sys

# Set the root directory to the repo root
# Get the current directory
current_dir = os.path.dirname(os.path.abspath('__file__'))
# Navigate to the repo root (assuming notebooks is directly under the repo root)
repo_root = os.path.abspath(os.path.join(current_dir, '..'))
# Add the repo root to the Python path
sys.path.insert(0, repo_root)
# Change the working directory to the repo root
os.chdir(repo_root)

In [ ]:
# Read the CSV file
df = pd.read_csv('combined_data/jan2025.csv')

# Display the first few rows of the dataframe
display(df.head(30))

In [ ]:
# Perform PCA on the dataframe
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# First, let's prepare the data for PCA
# Select only numerical columns (excluding the target variable 'fire')
features = df.drop(columns=['fire'])

# Standardize the features (important for PCA)
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# Apply PCA
pca = PCA()
pca_result = pca.fit_transform(scaled_features)

# Create a DataFrame with the principal components
pca_df = pd.DataFrame(
    data=pca_result,
    columns=[f'PC{i+1}' for i in range(pca_result.shape[1])]
)

# Display the explained variance ratio
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

# Plot the explained variance
plt.figure(figsize=(10, 6))
plt.bar(range(1, len(explained_variance) + 1), explained_variance, alpha=0.7, label='Individual explained variance')
plt.step(range(1, len(cumulative_variance) + 1), cumulative_variance, where='mid', label='Cumulative explained variance')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% explained variance threshold')
plt.xlabel('Principal Components')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Principal Components')
plt.legend()
plt.tight_layout()
plt.show()

# Display the first few principal components
display(pca_df.head())

# Show the feature loadings (correlation between original features and principal components)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.components_.shape[0])],
    index=features.columns
)
display(loadings)

# Determine how many components are needed to explain 95% of the variance
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1
print(f"Number of components needed to explain 95% of variance: {n_components_95}")


In [ ]:
# Now let's train a regression model to predict FRP

# First, let's prepare our data
# We'll use all features to predict FRP
X = features  # All features
y = df['frp']  # Target variable

# Check if there are any non-zero FRP values
print(f"Number of non-zero FRP values: {np.sum(y > 0)}")
print(f"Range of FRP values: {y.min()} to {y.max()}")

# Split the data into training and testing sets
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Let's try a few regression models

# 1. Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

linear_model = LinearRegression()
linear_model.fit(X_train, y_train)
y_pred_linear = linear_model.predict(X_test)

print("\nLinear Regression Results:")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred_linear)}")
print(f"R² Score: {r2_score(y_test, y_pred_linear)}")

# 2. Random Forest Regressor
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Results:")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred_rf)}")
print(f"R² Score: {r2_score(y_test, y_pred_rf)}")

# 3. XGBoost Regressor
import xgboost as xgb

xgb_model = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

print("\nXGBoost Results:")
print(f"Mean Squared Error: {mean_squared_error(y_test, y_pred_xgb)}")
print(f"R² Score: {r2_score(y_test, y_pred_xgb)}")

# Feature importance for the best model (assuming XGBoost performs best)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
})
feature_importance = feature_importance.sort_values('Importance', ascending=False)

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importance for FRP Prediction')
plt.tight_layout()
plt.show()
